In [ ]:
cryptos = {
    "BTCUSDT": "bitcoin",
    "ETHUSDT": "ethereum",
    "BNBUSDT": "binance",
    "SOLUSDT": "solana",
    "XRPUSDT": "ripple"
}
interval = "1h"
limit = 500
BASE_URL = "https://api.binance.com/api/v3/klines"
API_KEY = "0e30c2fd9f294f21b216b290a6edd700"

import joblib
model = joblib.load('../second_training/sentiment_model.pkl')
tfidf = joblib.load('../second_training/tfidf_vectorizer.pkl')

For getting price data from Binance.

In [ ]:
def get_price_data(symbol):
    params = {"symbol": symbol, "interval": interval, "limit": limit}
    response = requests.get(BASE_URL, params=params).json()
    data = pd.DataFrame(response, columns=[
        "timestamp", "open", "high", "low", "close", "volume",
        "close_time", "qav", "num_trades", "tbbav", "tbqav", "ignore"
    ])
    data["timestamp"] = pd.to_datetime(data["timestamp"], unit="ms")
    data["close"] = data["close"].astype(float)
    data["return"] = data["close"].pct_change()
    return data[["timestamp", "close", "return"]]

For getting news data from NewsApi.

In [ ]:
def fetch_newsapi_news(query="bitcoin", start_date="2025-10-01", end_date="2025-10-23"):
    base_url = "https://newsapi.org/v2/everything"
    all_articles = []
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    delta = timedelta(days=1)
    while start <= end:
        from_str = start.strftime("%Y-%m-%d")
        to_str = (start + delta).strftime("%Y-%m-%d")
        params = {
            "q": query,
            "from": from_str,
            "to": to_str,
            "language": "en",
            "sortBy": "publishedAt",
            "pageSize": 100,
            "apiKey": API_KEY
        }
        try:
            r = requests.get(base_url, params=params, timeout=10)
            data = r.json()
            if "articles" in data and data["articles"]:
                all_articles.extend(data["articles"])
        except Exception as e:
            print(f"Ошибка {e} на дате {from_str}")
        start += delta
        time.sleep(0.5)

    df = pd.DataFrame(all_articles)[["title", "description", "content", "timestamp"]]
    df["text"] = df[["title", "description", "content"]].astype(str).agg(". ".join, axis=1)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df.dropna(subset=["text", "timestamp"], inplace=True)
    return df

For getting news sentiment.

In [ ]:
def analyze_sentiment(df_news):
    X = tfidf.transform(df_news["text"])
    df_news["sentiment"] = model.predict(X)
    mapping = {"positive": 1, "neutral": 0, "negative": -1}
    df_news["sentiment"] = df_news["sentiment"].str.lower().map(mapping)
    print(df_news.columns)
    print(df_news["sentiment"].value_counts())
    return df_news

In [ ]:
def merged_data(df_news, df_price, lag_hours=3):
    df_news["timestamp"] = pd.to_datetime(df_news["timestamp"])
    df_news["timestamp_lagged"] = df_news["timestamp"] + pd.to_timedelta(lag_hours, unit="h")

    sentiment_hourly = df_news.groupby(df_news["timestamp_lagged"].dt.floor("H")["sentiment"].mean().reset_index()
    data_merge = pd.merge(df_price, sentiment_hourly, left_on="timestamp", right_index=True, how="left")
    data_merge["sentiment"].fillna(0, inplace=True)
    return data_merge

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

def correlation_analysis(df, symbol):
    corr_df = df.dropna(subset=["return", "sentiment"])
    pearson_corr, pearson_p = pearsonr(corr_df["sentiment"], corr_df["return"])
    spearman_corr, spearman_p = spearmanr(corr_df["sentiment"], corr_df["return"])
    print(f"Pearson correlation sentiment vs price: {pearson_corr:.3f}, p-value = {pearson_p:.3f}")
    print(f"Spearman correlation sentiment vs price: {spearman_corr:.3f}, p-value = {spearman_p:.3f}")

    sns.scatterplot(x="sentiment", y="return", data=df)
    plt.title(f"{symbol}: Корреляция (Пирсон={pearson_corr:.2f}, Спирмен={spearman_corr:.2f})")
    plt.xlabel("Тональность новостей (sentiment)")
    plt.ylabel("Изменение цены (return)")
    plt.show()

    return {
        "symbol": symbol,
        "pearson_corr": pearson_corr,
        "pearson_p": pearson_p,
        "spearman_corr": spearman_corr,
        "spearman_p": spearman_p
    }

In [2]:
def main():
    results = []
    for symbol, query in cryptos.items():
        print(f"Analisis for {symbol}:")
        price_df = get_price_data(symbol)
        news_df = fetch_newsapi_news(query=symbol, "2025-10-01", "2025-10-23")
        news_df = analyze_sentiment(news_df)
        merged = merged_data(news_df, price_df, lag_hours=3)
        merged_df.to_csv(f"merged_{symbol}.csv", index=False)
        result = correlation_analysis(merged_df, symbol)
        results.append({"crypt": symbol, "correlation": result})
    pd.DataFrame(results).to_csv("correlation_results.csv", index=False)

SyntaxError: positional argument follows keyword argument (1498488343.py, line 6)

In [1]:
if __name__ == "__main__":
    main()

NameError: name 'main' is not defined